In [ ]:
import io
import sys
import zipfile
import pandas as pd
import requests
from datetime import datetime
from dataclasses import dataclass
from typing import ClassVar

@dataclass
class BhavcopyConfigEquity:

    URL: ClassVar[dict[str, tuple[str, bool]]] = {
        "NSE": ("https://nsearchives.nseindia.com/content/cm/BhavCopy_NSE_CM_0_0_0_{Y}{MM}{DD}_F_0000.csv.zip", True),
        "BSE": ("https://www.bseindia.com/download/BhavCopy/Equity/BhavCopy_BSE_CM_0_0_0_{Y}{MM}{DD}_F_0000.CSV", False),
    }

    LEGACY_URL: ClassVar[dict[str, tuple[str, bool]]] = {
        "NSE": ("https://nsearchives.nseindia.com/content/historical/EQUITIES/{Y}/{MON}/cm{DD}{MON}{Y}bhav.csv.zip", True),
        "BSE": ("https://www.bseindia.com/download/BhavCopy/Equity/EQ_ISINCODE_{DD}{MM}{YY}.zip", True),
    }

    HEADERS: ClassVar[dict] = {
        "NSE": {"User-Agent": "Mozilla/5.0",
                "Referer": "https://www.nseindia.com/all-reports"},

        "BSE": {"User-Agent": "Mozilla/5.0",
                "Referer": "https://www.bseindia.com/markets/marketinfo/BhavCopy.aspx"},
    }


def get_bhavcopy_equity(date: str, exchange: str = "NSE", timeOut = 5) -> pd.DataFrame:

    legacy_end_date = datetime(2024, 7, 8)
    t_date = datetime.strptime(date, "%Y%m%d")
    exchange = exchange.upper()

    if exchange not in ['BSE', 'NSE']:
        raise ValueError('exchage must be NSE or BSE')

    url, iszip = (BhavcopyConfigEquity.URL if t_date >= legacy_end_date else BhavcopyConfigEquity.LEGACY_URL)[exchange]
    url = url.format(Y = t_date.year, YY = t_date.strftime("%y"), MM = t_date.strftime("%m"),\
                     DD = t_date.strftime("%d"), MON = t_date.strftime("%b").upper())

    ''' Data Loading '''
    s = requests.Session()
    s.headers.update(BhavcopyConfigEquity.HEADERS[exchange])
    if exchange == 'NSE':
        s.get("https://www.nseindia.com", timeout=timeOut)

    r = s.get(url, timeout=timeOut)
    r.raise_for_status()

    if iszip:
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            df = pd.read_csv(z.open(z.namelist()[0]))
    else:
        df = pd.read_csv(io.StringIO(r.text))

    
    """Data Processing"""
    if exchange == "BSE":
        
        if t_date >= legacy_end_date:
            columns = [
                "TradDt", "Src", "SctySrs", "FinInstrmTp", "ISIN", "TckrSymb",
                "FinInstrmNm", "OpnPric", "HghPric", "LwPric", "ClsPric",
                "LastPric", "SttlmPric", "TtlTradgVol", "TtlTrfVal",
                "TtlNbOfTxsExctd",
            ]
        else:
            columns = [
                "SC_CODE", "SC_NAME", "SC_GROUP", "ISIN_CODE", "OPEN", "HIGH",
                "LOW", "CLOSE", "LAST", "NO_TRADES", "NET_TURNOV",
            ]

        df = df[columns]

        if t_date >= legacy_end_date:
            df.columns = [
                "Date", "Exchange", "Instrument_Type", "Instrument_ID", "ISIN",
                "Ticker", "Name", "Open", "High", "Low", "Close", "Last",
                "Settlement", "Volume", "Turnover", "Trades",
            ]

        else:
            df.columns = [
                "Instrument_ID", "Name", "Instrument_Type", "ISIN", "Open",
                "High", "Low", "Close", "Last", "Trades", "Turnover",
            ]
            df["Date"] = t_date
            df["Exchange"] = exchange
            df["Settlement"] = df["Close"]

    elif exchange == "NSE":

        if t_date >= legacy_end_date:
            columns = [
                "TradDt", "Src", "SctySrs", "FinInstrmTp", "ISIN", "TckrSymb",
                "FinInstrmNm", "OpnPric", "HghPric", "LwPric", "ClsPric",
                "LastPric", "SttlmPric", "TtlTradgVol", "TtlTrfVal",
                "TtlNbOfTxsExctd",
            ]

            df = df[columns]
            df.columns = [
                "Date", "Exchange", "Instrument_Type", "Instrument_ID", "ISIN",
                "Ticker", "Name", "Open", "High", "Low", "Close", "Last",
                "Settlement", "Volume", "Turnover", "Trades",
            ]

        else:
            df = df[
                [
                    "TIMESTAMP", "SYMBOL", "SERIES", "OPEN", "HIGH", "LOW",
                    "CLOSE", "LAST", "TOTTRDQTY", "TOTTRDVAL", "TOTALTRADES", "ISIN",
                ]
            ]
            df.columns = [
                "Date", "Ticker", "Instrument_Type", "Open", "High", "Low",
                "Close", "Last", "Volume", "Turnover", "Trades", "ISIN",
            ]

            df["Date"] = t_date
            df["Exchange"] = exchange
            df["Settlement"] = df["Close"]
    
    df['Date'] = pd.to_datetime(df['Date'])
    return df

In [179]:
nse_old = get_bhavcopy_equity("20170102", "BSE")   # UDiFF
nse_new = get_bhavcopy_equity("20260102", "BSE")   # UDiFF


ReadTimeout: HTTPSConnectionPool(host='www.bseindia.com', port=443): Read timed out. (read timeout=5)

In [ ]:
nse_new

In [154]:
@dataclass
class BhavcopyConfigDerivatives:

    URL: ClassVar[dict[str, tuple[str, bool]]] = {
        "NSE": ("https://nsearchives.nseindia.com/content/fo/BhavCopy_NSE_FO_0_0_0_{Y}{MM}{DD}_F_0000.csv.zip", True),
        "BSE": ("https://www.bseindia.com/download/Bhavcopy/Derivative/BhavCopy_BSE_FO_0_0_0_{Y}{MM}{DD}_F_0000.CSV", False),
    }

    LEGACY_URL: ClassVar[dict[str, tuple[str, bool]]] = {
        "NSE": ("https://nsearchives.nseindia.com/content/historical/DERIVATIVES/{Y}/{MON}/fo{DD}{MON}{Y}bhav.csv.zip", True),
        "BSE": ("https://www.bseindia.com/download/Bhavcopy/Derivative/BSE_EQD_BHAVCOPY_{DD}{MM}{Y}.csv", False)
    }

    HEADERS: ClassVar[dict] = {
        "NSE": {"User-Agent": "Mozilla/5.0",
                "Referer": "https://www.nseindia.com/all-reports-derivatives"},

        "BSE": {"User-Agent": "Mozilla/5.0",
                "Referer": "https://www.bseindia.com/markets/marketinfo/BhavCopy.aspx"},
    }


In [171]:
def get_bhavcopy_derivative(date: str, exchange: str = "NSE", timeOut = 5) -> pd.DataFrame:

    legacy_end_date = datetime(2024, 7, 8)
    t_date = datetime.strptime(date, "%Y%m%d")
    exchange = exchange.upper()

    if exchange not in ['BSE', 'NSE']:
        raise ValueError('exchage must be NSE or BSE')

    url, iszip = (BhavcopyConfigDerivatives.URL if t_date >= legacy_end_date else BhavcopyConfigDerivatives.LEGACY_URL)[exchange]
    url = url.format(Y = t_date.year, YY = t_date.strftime("%y"), MM = t_date.strftime("%m"),\
                     DD = t_date.strftime("%d"), MON = t_date.strftime("%b").upper())

    ''' Data Loading '''
    s = requests.Session()
    s.headers.update(BhavcopyConfigDerivatives.HEADERS[exchange])
    if exchange == 'NSE':
        s.get("https://www.nseindia.com", timeout=timeOut)

    r = s.get(url, timeout=timeOut)
    r.raise_for_status()

    if iszip:
            with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                filename = z.namelist()[0]
    
                with z.open(filename) as f:
                    if filename.lower().endswith((".xls", ".xlsx")):
                        df = pd.read_excel(f, header = 1)
                    else:
                        df = pd.read_csv(f)

    else:
        df = pd.read_csv(io.StringIO(r.text))

    return df

In [172]:
bse_old = get_bhavcopy_equity("20170102", "BSE")   # UDiFF
bse_new = get_bhavcopy_equity("20261030", "BSE")   # UDiFF

nse_old = get_bhavcopy_equity("20170102", "NSE")   # UDiFF
nse_new = get_bhavcopy_equity("20261030", "NSE")   # UDiFF

bse_old.to_csv('BSE_Old.csv')
bse_new.to_csv('BSE_New.csv')

nse_old.to_csv('NSE_Old.csv')
nse_new.to_csv('NSE_New.csv')

In [173]:
@dataclass
class BhavcopyConfigDerivatives:

    URL: ClassVar[dict[str, tuple[str, bool]]] = {
        "NSE": ("https://nsearchives.nseindia.com/content/fo/BhavCopy_NSE_FO_0_0_0_{Y}{MM}{DD}_F_0000.csv.zip", True),
        "BSE": ("https://www.bseindia.com/download/Bhavcopy/Derivative/BhavCopy_BSE_FO_0_0_0_{Y}{MM}{DD}_F_0000.CSV", False),
    }

    LEGACY_URL: ClassVar[dict[str, tuple[str, bool]]] = {
        "NSE": ("https://nsearchives.nseindia.com/content/historical/DERIVATIVES/{Y}/{MON}/fo{DD}{MON}{Y}bhav.csv.zip", True),
        "BSE": ("https://www.bseindia.com/download/BhavCopy/Derivative/bhavcopy{DD}-{MM}-{YY}.zip", True)
    }

    HEADERS: ClassVar[dict] = {
        "NSE": {"User-Agent": "Mozilla/5.0",
                "Referer": "https://www.nseindia.com/all-reports-derivatives"},

        "BSE": {"User-Agent": "Mozilla/5.0",
                "Referer": "https://www.bseindia.com/markets/marketinfo/BhavCopy.aspx"},
    }


In [174]:
bse_new = get_bhavcopy_derivative("20260102", "BSE") 
bse_old = get_bhavcopy_derivative("20170102", "BSE") 

nse_new = get_bhavcopy_derivative("20260102", "NSE") 
nse_old = get_bhavcopy_derivative("20170102", "NSE") 

bse_old.to_csv('BSE_Options_Old.csv')
bse_new.to_csv('BSE_Options_New.csv')

nse_old.to_csv('NSE_Options_Old.csv')
nse_new.to_csv('NSE_Options_New.csv')


In [175]:
bse_new.columns

Index(['TradDt', 'BizDt', 'Sgmt', 'Src', 'FinInstrmTp', 'FinInstrmId', 'ISIN',
       'TckrSymb', 'SctySrs', 'XpryDt', 'FininstrmActlXpryDt', 'StrkPric',
       'OptnTp', 'FinInstrmNm', 'OpnPric', 'HghPric', 'LwPric', 'ClsPric',
       'LastPric', 'PrvsClsgPric', 'UndrlygPric', 'SttlmPric', 'OpnIntrst',
       'ChngInOpnIntrst', 'TtlTradgVol', 'TtlTrfVal', 'TtlNbOfTxsExctd',
       'SsnId', 'NewBrdLotQty', 'Rmks', 'Rsvd1', 'Rsvd2', 'Rsvd3', 'Rsvd4'],
      dtype='str')

In [176]:
bse_new

,TradDt,BizDt,Sgmt,Src,FinInstrmTp,FinInstrmId,ISIN,TckrSymb,SctySrs,XpryDt,...,TtlTradgVol,TtlTrfVal,TtlNbOfTxsExctd,SsnId,NewBrdLotQty,Rmks,Rsvd1,Rsvd2,Rsvd3,Rsvd4
0,2026-01-02,2026-01-02,FO,BSE,IDO,1165809,NaN,SENSEX,NaN,2026-01-15,...,25040,2.105086e+09,854,F1,20,NaN,NaN,NaN,NaN,NaN
1,2026-01-02,2026-01-02,FO,BSE,IDO,1165318,NaN,SENSEX,NaN,2026-01-15,...,2780,2.411047e+08,111,F1,20,NaN,NaN,NaN,NaN,NaN
2,2026-01-02,2026-01-02,FO,BSE,IDO,1160417,NaN,SENSEX,NaN,2026-01-08,...,6200,5.604986e+08,270,F1,20,NaN,NaN,NaN,NaN,NaN
3,2026-01-02,2026-01-02,FO,BSE,IDO,1159964,NaN,SENSEX,NaN,2026-01-08,...,6218060,5.357041e+11,128934,F1,20,NaN,NaN,NaN,NaN,NaN
4,2026-01-02,2026-01-02,FO,BSE,IDO,1150959,NaN,SENSEX,NaN,2026-01-29,...,740,6.516528e+07,33,F1,20,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
586,2026-01-02,2026-01-02,FO,BSE,IDO,1160014,NaN,SENSEX,NaN,2026-01-08,...,8220,6.625805e+08,290,F1,20,NaN,NaN,NaN,NaN,NaN
587,2026-01-02,2026-01-02,FO,BSE,IDO,1159923,NaN,SENSEX,NaN,2026-01-08,...,6180,4.900967e+08,86,F1,20,NaN,NaN,NaN,NaN,NaN
588,2026-01-02,2026-01-02,FO,BSE,IDO,1163270,NaN,SENSEX,NaN,2026-01-15,...,15620,1.357947e+09,500,F1,20,NaN,NaN,NaN,NaN,NaN
589,2026-01-02,2026-01-02,FO,BSE,IDO,1160224,NaN,SENSEX,NaN,2026-01-08,...,1223740,1.138099e+11,6894,F1,20,NaN,NaN,NaN,NaN,NaN
